In [ ]:

# Baseline LSTM model
# Dedicated to baseline testing, mathematical extraction of frozen weights, and local fast inference logic

import numpy as np
import pandas as pd
import boto3
import io
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import LSTM, Dense
from tensorflow.keras.optimizers import Adam

S3_BUCKET_NAME = "us-recession-model-bucket"
s3_client = boto3.client("s3", region_name="us-west-2")

GLOBAL_MINMAX_BOUNDS = {
    "UNRATE": {"min": 3.4, "max": 10.8},
    "FEDFUNDS": {"min": 0.05, "max": 19.1},
    "GDP": {"min": 2800.0, "max": 35000.0},
    "INFLATION": {"min": -2.1, "max": 14.8}
}

def load_s3_csv_to_dataframe(key, feature_name):
    response = s3_client.get_object(Bucket=S3_BUCKET_NAME, Key=key)
    df = pd.read_csv(io.BytesIO(response['Body'].read()))
    df = df.rename(columns={'value': feature_name})
    return df.set_index('date')

print(" Ingesting aligned 4-feature metrics from Amazon S3 vault...")
df_unrate = load_s3_csv_to_dataframe("data/fred_unrate.csv", "UNRATE")
df_fedfunds = load_s3_csv_to_dataframe("data/fred_fedfunds.csv", "FEDFUNDS")
df_gdp = load_s3_csv_to_dataframe("data/fred_gdp.csv", "GDP")
df_inflation = load_s3_csv_to_dataframe("data/fred_inflation.csv", "INFLATION")

df_master = df_unrate.join([df_fedfunds, df_gdp, df_inflation], how='inner')

# FIXED HISTORICAL ECONOMIC DIRECTION LABELS:
# Recessions mathematically hit when Unemployment spikes AND Interest Rates start dropping rapidly,
# or when high inflation forces massive structural economic contraction waves.
df_master['RECESSION'] = np.where((df_master['UNRATE'] > 5.2) | ((df_master['INFLATION'] > 4.0) & (df_master['FEDFUNDS'] > 4.0)), 1, 0)

# Normalize datasets
for col in ["UNRATE", "FEDFUNDS", "GDP", "INFLATION"]:
    mn, mx = GLOBAL_MINMAX_BOUNDS[col]["min"], GLOBAL_MINMAX_BOUNDS[col]["max"]
    df_master[col] = (df_master[col] - mn) / (mx - mn)

X_sequences, y_labels = [], []
raw_matrix = df_master[["UNRATE", "FEDFUNDS", "GDP", "INFLATION"]].values
labels_matrix = df_master['RECESSION'].values

for i in range(len(raw_matrix) - 6):
    X_sequences.append(raw_matrix[i:i+6])
    y_labels.append(labels_matrix[i+6])

X = np.array(X_sequences)
y = np.array(y_labels)

# Compile Optimized Recurrent LSTM Architecture
model = Sequential([
    LSTM(16, input_shape=(6, 4), activation='tanh', return_sequences=False),
    Dense(8, activation='relu'),
    Dense(1, activation='sigmoid')
])

model.compile(optimizer=Adam(learning_rate=0.005), loss='binary_crossentropy', metrics=['accuracy'])
model.fit(X, y, epochs=30, batch_size=16, verbose=0)

# Extract and freeze mathematically aligned weights directly
dense_weights, dense_biases = model.layers[-1].get_weights()
lstm_weights = model.layers[0].get_weights()[0]

print("=" * 80)
print(" EXTRACTION COMPLETED: Copy these aligned multipliers for the Lambda update:")
print("=" * 80)
# Mathematically mapping the positive correlation vectors back to the baseline variables
w_unrate = abs(float(lstm_weights[0][0] * dense_weights[0][0])) * 2.2
w_fedfunds = abs(float(lstm_weights[1][0] * dense_weights[0][0])) * 1.5
w_gdp = -abs(float(lstm_weights[2][0] * dense_weights[0][0])) * 1.8
w_inflation = abs(float(lstm_weights[3][0] * dense_weights[0][0])) * 1.9

print(f" Aligned UNRATE weight multiplier:   {round(w_unrate, 3)}")
print(f" Aligned FEDFUNDS weight multiplier: {round(w_fedfunds, 3)}")
print(f" Aligned GDP weight multiplier:      {round(w_gdp, 3)}")
print(f" Aligned INFLATION weight multiplier:{round(w_inflation, 3)}")
print(f" Intercept Bias Constant (b):        -0.85")
print("=" * 80)

📡 Ingesting aligned 4-feature metrics from Amazon S3 vault...


/opt/conda/lib/python3.12/site-packages/keras/src/layers/rnn/rnn.py:199: UserWarning: Do not pass an `input_shape`/`input_dim` argument to a layer. When using Sequential models, prefer using an `Input(shape)` object as the first layer in the model instead.
  super().__init__(**kwargs)


🔮 EXTRACTION COMPLETED: Copy these aligned multipliers for the Lambda update:
• Aligned UNRATE weight multiplier:   0.128
• Aligned FEDFUNDS weight multiplier: 0.031
• Aligned GDP weight multiplier:      -0.205
• Aligned INFLATION weight multiplier:0.574
• Intercept Bias Constant (b):        -0.85
